# ⚠️ Remaining Issues:
1. GED (in RT dist measure)
*        needs fixing: internal nodes mismatching effects the score in unexpected ways.
2. TBD

# Imports

In [ ]:
from reticulate_tree import ReticulateTree
import compare_reticulations as cr
import importlib
import pandas as pd


# Example usages of ReticulateTree

In [ ]:
# Test the enewick input case:
test_enewick = '((a,(e)#H1:0),(#H1:0,b));'
# '((KBNS10,(SRR27702572,(SRR27702568,(((((KPIS19,(KSFS35)#H8),(KMTS31)#H6),(KRBS33)#H2),(KHUS06,(PZ382,(KCOS43,(KGUS05,(KPXS21)#H0))))),(KCUS03,(KVNS23)#H4))))),((((#H6,#H8),#H2),#H4),#H0));'
obj = ReticulateTree(test_enewick)
print(f'Parsed ETE3 tree from enriched Newick: {obj.tree_str}\n')
obj.visualize()
print(obj.to_enewick())


# Debug and usage of some compare funcs

In [ ]:
input_A = {'a':2,'b':2,'c':1,'d':1,'e':1}
input_B = {'a':1,'b':2,'c':3,'d':1,'e':1}
res = cr.compare_ploidy_diff(input_A, input_B)
print(f'Ploidy Jaccard test result: {res}')
# We expect: {'dist': 0.75, 'FP': 2, 'FN': 1, 'TP': 1}
# out of 4 extra copies (1+1+2+0+0), 2 is correct (from b), 2 are false positive (both from c), 1 is false negative (from a)
# therefore the distance is 0.75 i.e., 3/4 differ = 1 - (1/4 match)

In [ ]:
input_lA = {'1': ['z', 'y', 'x'], '2': ['w']}
input_sA = {'1': [{'a'}, {'b'}], '2': [{'c'}, {'d'}]}

input_lB = {'3': ['x', 'y', 'w', 'z'], '4': ['w']}
input_sB = {'3': [{'d', 'w', 'c'}, {'a'}], '4': [{'c'}, {'y', 'z'}]}

res = cr.match_and_compare(input_lA, input_lB)
print(f'Reticulation leaves test result: {res}')

res = cr.match_and_compare(input_lA, input_lB, input_sA, input_sB)
print(f'Reticulation sisters test result: {res}')

In [ ]:
input1, input2 = {1:{'w', 'y'}, 2:{'c'}, 3:{'u', 'a', 'b'}, 4:{'z'}}, {1:{'y', 'z', 'x'}, 2:{'c'}, 3:{'a', 'b'}, 4:{'y', 'z'}, 5:{'u'}}

#input1, input2 = [{'c'}, {'u', 'a', 'b'}], [{'c'}, {'a', 'b'}, {'u'}]

cr.match_and_compare(input1, input2, debug=True)

# should u match to a,b,u too?

In [ ]:
input_lA = {'1': ['z', 'y', 'x'], '2': ['w'], '3': ['u', 'v']}
input_sA = {'1': [{'a'}, {'b'}], '2': [{'c'}, {'d', 'y'}], '3': [{'u'}, {'v'}]}

input_lB = {'3': ['x', 'y', 'w', 'z'], '4': ['w']}
input_sB = {'3': [{'d', 'w', 'c'}, {'a'}, {'u'}], '4': [{'c'}, {'y', 'z'}]}

res = cr.match_and_compare(input_lA, input_lB, input_sA, input_sB, debug=True)
print(f'Reticulation sisters test result: {res}')

# Refresh lib

In [ ]:
importlib.reload(cr)

# Run all step by step

In [ ]:
# Test a family of MUL-trees derived from the same single-labeled tree (all kinds of reticulation placements):
phylo_trees = {
    'no_reticules': '(((c,w),d)Q,((a,(x,(y,z))),b));',
    'no_intercladic': '(((c,w),(d,w)),((a,(x,(y,z))),(b,(x,(y,z)))));',
    'no_nested': '((((c,w),(d,w))Q,(x,(y,z))),((a,(x,(y,z))),b));',
    'semi_nested': '((((c,w),d),(x,((y,z),w))),((a,(x,((y,z),w))),b));',
    'once_nested': '(((c,d),((x,w),((y,z),w))),((a,((x,w),((y,z),w))),b));',
    'semi_broken1': '((((c,w),d),(x,(y,z))),((a,(x,((y,z),w))),b));',
    'semi_broken2': '((((c,w),d),(x,((y,z),w))),((a,(x,(y,z))),b));',
    'semi_shifted1': '((((c,w),d),(x,(y,(z,w)))),((a,(x,((y,z),w))),b));',
    'semi_shifted2': '((((c,w),d),(x,((y,z),w))),((a,((x,(y,z)),w)),b));',
}

phylo_trees_data = []
for name, newick_str in phylo_trees.items():

    # MUL-tree string parsing tests
    phylo_trees[name] = ReticulateTree(newick_str)
    curr_obj = phylo_trees[name]
    print(f'Name: {name} - newick: {curr_obj.tree_str}')

    # Back-conversion test:
    dag = curr_obj.dag
    print(f'Nodes: {dag.number_of_nodes()}, Edges: {dag.number_of_edges()}')
    back_tree = ReticulateTree.dag_to_multree(dag)
    #print(f'Converted back to tree: {back_tree.write(format=8)}')

    # Check if the back-converted tree matches the original
    print(f'Is back-converted tree identical to original? {cr.are_identical(curr_obj, ReticulateTree(back_tree))}')

    # Measure the object and check printout
    object_measurements = curr_obj.measure(printout=True)

    phylo_trees_data.append({
        'name': name,
        'object': curr_obj,
        **object_measurements
    })

    # Visualize the object
    curr_obj.visualize()#main_path / f'{name}_dag.png')

    # Interactive visualization (if needed)
    #curr_obj.interact(main_path / f'{name}_interactive.html', launch=True)

    print('\n')

In [ ]:
# Create a DataFrame for the phylogenetic trees data
phylo_df = pd.DataFrame(phylo_trees_data).set_index('name')

In [ ]:
# Generate pairwise comparison matrices
comparison_matrices = cr.pairwise_comparison(phylo_df)

for metric, matrix in comparison_matrices.items():
    print(f'\n{metric.upper()} matrix:')
    print(matrix.round(3))

In [ ]:
cr.plot_comparison_heatmaps(comparison_matrices, phylo_df)

# One stop shop

In [ ]:
phylo_trees = {
    'no_reticules': '(((c,w),d)Q,((a,(x,(y,z))),b));',
    'no_intercladic': '(((c,w),(d,w)),((a,(x,(y,z))),(b,(x,(y,z)))));',
    'no_nested': '((((c,w),(d,w))Q,(x,(y,z))),((a,(x,(y,z))),b));',
    'semi_nested': '((((c,w),d),(x,((y,z),w))),((a,(x,((y,z),w))),b));',
    'once_nested': '(((c,d),((x,w),((y,z),w))),((a,((x,w),((y,z),w))),b));',
    'semi_broken1': '((((c,w),d),(x,(y,z))),((a,(x,((y,z),w))),b));',
    'semi_broken2': '((((c,w),d),(x,((y,z),w))),((a,(x,(y,z))),b));',
    'semi_shifted1': '((((c,w),d),(x,(y,(z,w)))),((a,(x,((y,z),w))),b));',
    'semi_shifted2': '((((c,w),d),(x,((y,z),w))),((a,((x,(y,z)),w)),b));',
}

results = cr.one_stop_compare(phylo_trees, printout=False)

In [ ]:
cr.plot_comparison_heatmaps(results['comparisons'], results['data'])